# Daniel Graham Model Testing
## xboost & KNN
Dataset
https://archive.ics.uci.edu/dataset/2/adult 

Environment based on Anaconda (python >= 3.13 and having install xgboost `conda install xgboost`)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.impute import KNNImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier # type: ignore


In [2]:
all_columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 
    'capital-loss', 'hours-per-week', 'native-country', 'income']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
target_feature = 'income'
random_state_value = 48

df = pd.read_csv('adult.data.csv', header=None, names=all_columns)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
X = df[categorical_features + numerical_features]
y = df[target_feature].apply(lambda x: 1 if x == ' >50K' else 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state_value, stratify=y)

categorical_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
numerical_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [4]:
xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)

xgb_model = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", xgb)])
xgb_model.fit(X_train, y_train)
predictions = xgb_model.predict(X_test)
predictions_proba = xgb_model.predict_proba(X_test)[:, 1]

In [5]:
confusion = confusion_matrix(y_test, predictions)
print("Confusion Matrix:")
print(confusion)

Confusion Matrix:
[[4641  304]
 [ 506 1062]]


In [6]:


print("\nClassification Report:")
print(classification_report(y_test, predictions))
print("\nPredicted Probabilities:")
print(predictions_proba)
print("Accuracy:", np.mean(predictions == y_test.values))
print("Precision:", confusion[1, 1] / (confusion[0, 1] + confusion[1, 1]))
print("Recall:", confusion[1, 1] / (confusion[1, 0] + confusion[1, 1]))


Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4945
           1       0.78      0.68      0.72      1568

    accuracy                           0.88      6513
   macro avg       0.84      0.81      0.82      6513
weighted avg       0.87      0.88      0.87      6513


Predicted Probabilities:
[0.0080122  0.00662933 0.01188828 ... 0.01720138 0.7144648  0.04123841]
Accuracy: 0.8756333486872409
Precision: 0.7774524158125915
Recall: 0.6772959183673469


In [ ]:
# KNN model using the same preprocessing pipeline as the XGBoost model
knn = KNeighborsClassifier(n_neighbors=5, weights='uniform')
knn_model = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', knn)])
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

knn_confusion = confusion_matrix(y_test, knn_predictions)
print('KNN Confusion Matrix:')
print(knn_confusion)
print('\nKNN Classification Report:')
print(classification_report(y_test, knn_predictions))
print('\nKNN Predicted Probabilities:')
print(knn_probabilities)
print('KNN Accuracy:', np.mean(knn_predictions == y_test.values))
print('KNN Precision:', knn_confusion[1, 1] / (knn_confusion[0, 1] + knn_confusion[1, 1]))
print('KNN Recall:', knn_confusion[1, 1] / (knn_confusion[1, 0] + knn_confusion[1, 1]))
